#### making policy/reward

In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from sim_class import Simulation
import pybullet as p


class OT2Env(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 60}

    def __init__(self, use_gui=False, max_steps=1000):
        super().__init__()

        self.use_gui = use_gui
        self.max_steps = max_steps
        self.steps = 0

        # --- PyBullet connection mode ---
        self.connection_mode = p.GUI if use_gui else p.DIRECT

        # Start clean physics client
        self.client_id = p.connect(self.connection_mode)

        # Simulation
        self.sim = Simulation(num_agents=1)

        # Action space: velocity command
        self.action_space = spaces.Box(
            low=-1, high=1, shape=(3,), dtype=np.float32
        )

        # Observation: pipette pos + goal pos
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(6,), dtype=np.float32
        )

        # Workspace bounds
        self.x_bounds = (-0.187, 0.253)
        self.y_bounds = (-0.171, 0.220)
        self.z_bounds = (0.168, 0.290)

        # Initial goal
        self.goal_position = np.zeros(3, dtype=np.float32)

    def _extract_pos(self, state):
        robot_key = [k for k in state.keys() if "robotId" in k][0]
        return np.array(state[robot_key]["pipette_position"], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        if seed is not None:
            np.random.seed(seed)

        # Random goal
        self.goal_position = np.array([
            np.random.uniform(*self.x_bounds),
            np.random.uniform(*self.y_bounds),
            np.random.uniform(*self.z_bounds)
        ], dtype=np.float32)

        # Reset simulation
        obs = self.sim.reset(num_agents=1)
        pipette_pos = self._extract_pos(obs)

        self.steps = 0
        observation = np.concatenate([pipette_pos, self.goal_position])
        return observation, {}

    def step(self, action):
        # Add dummy "drop" value
        action_full = np.append(action, 0)

        obs = self.sim.run([action_full])
        pipette_pos = self._extract_pos(obs)

        observation = np.concatenate([pipette_pos, self.goal_position])

        # Distance-based reward
        dist = np.linalg.norm(pipette_pos - self.goal_position)

        reward = -dist

        terminated = dist < 0.01  # success
        truncated = self.steps >= self.max_steps

        if terminated:
            reward += 100.0  # goal bonus

        self.steps += 1

        return observation, reward, terminated, truncated, {}

    def render(self):
        pass  # PyBullet GUI handles rendering automatically

    def close(self):
        if p.isConnected(self.client_id):
            p.disconnect(self.client_id)


#### testing env

In [2]:
from stable_baselines3.common.env_checker import check_env
#from ot2_env_wrapper import OT2Env
import numpy as np

# First, check if the environment is valid
print("Checking environment...")
env = OT2Env(render=False)
check_env(env)
print("Environment check passed!")

# Now run some test episodes
print("\nRunning test episodes...")
num_episodes = 5

for episode in range(num_episodes):
    obs, info = env.reset()
    done = False
    step = 0
    total_reward = 0

    print(f"\n--- Episode {episode + 1} ---")
    print(f"Starting position: {obs[:3]}")
    print(f"Goal position: {obs[3:]}")

    while not done:
        # Take a random action from the environment's action space
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        
        total_reward += reward
        done = terminated or truncated

        step += 1
        
        if done:
            print(f"\nEpisode finished after {step} steps")
            print(f"Total reward: {total_reward:.2f}")
            print(f"Terminated: {terminated}, Truncated: {truncated}")
            break

env.close()
print("\nTesting complete!")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Checking environment...


TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'

#### training model

In [2]:
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
#from OT2Env import OT2Env

def make_env():
    return OT2Env(use_gui=False)
# Vectorized environment (single env)
env = DummyVecEnv([make_env])

model = SAC(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    gamma=0.99,
    buffer_size=200000,
    batch_size=256,
)

model.learn(total_timesteps=200_000)
model.save("ot2_sac_model")
env.close()


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cuda device


KeyboardInterrupt: 

#### testing model

In [ ]:
import numpy as np
from stable_baselines3 import PPO
#from ot2_env import OT2Env   # <-- Your environment file
import time
import matplotlib.pyplot as plt

# -----------------------------------------
# Load environment and model
# -----------------------------------------
env = OT2Env()
model = PPO.load(r'C:\Users\koenm\Documents\robotics_env\models\ot2_model_260000_steps.zip')

# -----------------------------------------
# Run one evaluation episode
# -----------------------------------------
obs, _ = env.reset()
print("Starting evaluation…\n")

for step in range(200):

    action, _ = model.predict(obs, deterministic=True)

    # Step safely
    obs, reward, done, truncated, info = env.step(action)

    pip = obs[:3]
    goal = obs[3:]
    dist = info.get("distance", np.linalg.norm(pip - goal))

    # NaN / explosion guard
    if np.any(np.isnan(pip)):
        print(" NaN detected at step", step)
        break

    print(f"{step:03d} | Pipette: {pip} | Goal: {goal} | Dist: {dist:.5f}")

    if done:
        print("\n Goal reached! Episode finished.")
        break

    if truncated:
        print("\n Episode truncated (max steps).")
        break

env.close()
print("\nEvaluation complete.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\koenm\miniconda3\envs\drone_env\lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU

Starting evaluation…

000 | Pipette: [0.0725 0.0889 0.1205] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.20742
001 | Pipette: [0.0716 0.0878 0.1224] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.20559
002 | Pipette: [0.0703 0.0862 0.1253] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.20290
003 | Pipette: [0.0684 0.0839 0.1291] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.19922
004 | Pipette: [0.0661 0.0817 0.1333] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.19506
005 | Pipette: [0.0634 0.0796 0.1405] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.18934
006 | Pipette: [0.0602 0.0777 0.1464] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.18386
007 | Pipette: [0.0565 0.0759 0.1511] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.17848
008 | Pipette: [0.0524 0.0743 0.1552] | Goal: [-0.10777839  0.06184343  0.21944341] | Dist: 0.17303
009 | Pipette: [0.0482 0.0728 0.1594] | Goal: [-0.10777839  0.06184343  0.2194